# 201 · Compression is not a format

Companion to [Compression vs format](https://leo-gan.github.io/GLD.SerializerBenchmark/theory/201/compression-is-not-a-format/).

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/leo-gan/GLD.SerializerBenchmark/blob/master/docs/theory/notebooks/201/compression_vs_format.ipynb)

**Goal:** separate serialization density from gzip/zlib compression; see CPU vs size trade-offs on a verbose payload.

**Why this lab:** enabling gzip often hides a verbose format and creates the myth that “serialization no longer matters.”

**How to use:** build verbose JSON and a dense toy encoding of the same rows; compress both with zlib; compare size and median compress time.

**Expect:** JSON compresses with a high ratio (lots of repeated keys); dense raw is already smaller; compressed dense may still win on absolute size with less compress work.

> **Honesty banner:** sizes and timings here are **illustrative**. Suite [Results](https://leo-gan.github.io/GLD.SerializerBenchmark/) own harness truth. Compare within one language and paradigm—not global format rankings.


## Build a verbose payload and a denser encoding

**Why:** compressors love repeated keys and text redundancy—common in JSON logs/events.

**How:** 200 wide-ish records as compact JSON; same logical columns in a toy dense binary without repeating key strings.

**Expect:** JSON many KB; dense encoding a few KB before any compression.

**Why it matters:** format-aware density removes redundancy *using the data model*; gzip removes whatever entropy remains afterward.


In [ ]:
import json
import time
import zlib
from statistics import median

# Verbose JSON-like records with repeated keys (why compressors love JSON)
rows = [
    {
        "sensor_id": f"plant-zone-{i % 7}",
        "temperature_celsius": 20 + (i % 10) * 0.1,
        "relative_humidity_percent": 40 + (i % 20),
        "status_label": "ok" if i % 5 else "warn",
    }
    for i in range(200)
]
raw_json = json.dumps(rows, separators=(",", ":")).encode("utf-8")
print("uncompressed JSON nbytes:", len(raw_json))



In [ ]:
def encode_varint(u: int) -> bytes:
    out = bytearray()
    while u > 0x7F:
        out.append((u & 0x7F) | 0x80)
        u >>= 7
    out.append(u & 0x7F)
    return bytes(out)


def encode_dense(rows: list) -> bytes:
    # toy columnar-ish: count + columns without repeating key strings
    out = bytearray()
    out += encode_varint(len(rows))
    for r in rows:
        sid = r["sensor_id"].encode()
        out += encode_varint(len(sid)) + sid
        # float as int milli-degrees for density demo
        out += encode_varint(int(r["temperature_celsius"] * 1000))
        out += encode_varint(int(r["relative_humidity_percent"]))
        out += encode_varint(0 if r["status_label"] == "ok" else 1)
    return bytes(out)


dense = encode_dense(rows)
print("dense toy format nbytes:", len(dense))



## Compress both; compare size and CPU

**Why:** compression is a **semantics-agnostic** transform on bytes; serialization defines meaning.

**How:** `zlib.compress` on JSON bytes and dense toy bytes; record median wall time.

**Expect:** large JSON → much smaller after zlib; dense format starts smaller; ratios differ; CPU time is nonzero either way.

**Why it matters:** on small RPCs compression can hurt latency; on bulk storage it often helps—but it never replaces schema evolution or typed access.


In [ ]:
def bench_compress(data: bytes, level: int = 6, repeat: int = 7):
    times = []
    out = b""
    for _ in range(repeat):
        t0 = time.perf_counter()
        out = zlib.compress(data, level=level)
        times.append(time.perf_counter() - t0)
    return out, median(times)


j_z, j_t = bench_compress(raw_json)
d_z, d_t = bench_compress(dense)

print(f"{'layer':30} {'nbytes':>8}  {'median compress s':>18}")
print(f"{'JSON raw':30} {len(raw_json):8}")
print(f"{'JSON + zlib':30} {len(j_z):8}  {j_t:18.6f}")
print(f"{'dense raw':30} {len(dense):8}")
print(f"{'dense + zlib':30} {len(d_z):8}  {d_t:18.6f}")
print()
print("JSON compression ratio:", round(len(raw_json) / len(j_z), 2))
print("dense compression ratio:", round(len(dense) / max(len(d_z), 1), 2))
print(
    "Note: verbose JSON often compresses *more* (ratio), yet dense raw may still "
    "be smaller before compress and cheaper at small messages."
)



## Mental model

```text
values → serialization (meaning) → optional compression (opaque smaller blob)
```

**Why this layering:** each stage answers a different question—contract vs redundancy.

Compression does **not** provide schema evolution, types, or field access.

**Expect:** after compress, tools see an opaque blob until decompress; field-level pushdown needs format-aware layouts (e.g. columnar pages).


## Takeaways

1. Pick format for contract + access pattern first.
2. Then measure compression at the right tier (link, file, column page).
3. Small RPCs: compression may hurt latency; large batches: often win.

**Why it matters:** “just gzip it” is an incomplete architecture decision.

**Back to 201 index:** [Serialization 201](https://leo-gan.github.io/GLD.SerializerBenchmark/theory/201/)
